# 📰 News Embedding Worker
Embeds news articles using **Qwen3-Embedding-8B** on Kaggle T4 GPU and saves vectors to PostgreSQL via n8n webhooks.

---
### ⚠️ Before running this notebook:
1. Attach the `qwen3-embedding-8b-model` dataset (right panel → Input → Add Input)
2. Set Accelerator to **GPU T4 x2** (right panel)
3. Turn Internet **ON** (right panel)
4. Fill in your credentials in **Cell 2** below
5. Run all cells top to bottom: **Cell 1 → 2 → 3 → 4**
---

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# Run once per session. Takes ~1 minute.
# ============================================================
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers", "psutil"])

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("✅ Dependencies installed")

---
## ✏️ Cell 2 — Fill in your credentials here

In [ ]:
# ============================================================
# CELL 2 — Configuration
# Edit the values below before running.
# ============================================================
import os
import glob
import socket
import platform
import psutil
from collections import deque
from datetime import datetime, timezone

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# --- n8n webhook URLs (get these from your n8n instance) ---
N8N_GET_URL  = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0265"     # e.g. https://n8n.yourdomain.com/webhook/xxx
N8N_SAVE_URL = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0255"  # e.g. https://n8n.yourdomain.com/webhook/yyy
N8N_STATUS_URL = "https://n8n.3rfan.ir/webhook/worker-status"                    # For sending progress updates / heartbeat; set to None to disable
N8N_API_KEY  = "mer30kehasti"               # the X-API-Key secret set in n8n
N8N_HEADERS  = {"X-API-Key": N8N_API_KEY, "Content-Type": "application/json"}

# --- Worker identity (use a unique name if running multiple workers) ---
NODE_NAME    = "kaggle-t4-worker"
MODEL_NAME   = "Qwen/Qwen3-Embedding-8B"

# --- Performance settings ---
BATCH_SIZE   = 1      # articles fetched per cycle (keep at 1 to avoid OOM)
MAX_HOURS    = 8.5    # stop before Kaggle's 9h session limit
STATUS_UPDATE_INTERVAL = 100  # Send status update every N articles (configurable, default 100)

# --- Worker heartbeat and status reporting ---
SESSION_STARTED_AT = datetime.now(timezone.utc)
EMBEDDING_TIMESTAMPS = deque()

# --- Model path (do not change if dataset is attached correctly) ---
matches = glob.glob("/kaggle/input/**/qwen3-embedding", recursive=True)
if matches:
    MODEL_PATH = matches[0]
    print(f"✅ Model found at: {MODEL_PATH}")
else:
    MODEL_PATH = "/kaggle/input/datasets/YOUR_USERNAME/qwen3-embedding-8b-model/qwen3-embedding"
    print(f"⚠️  Model not auto-detected. Set MODEL_PATH manually: {MODEL_PATH}")


def set_n8n_status_url(url):
    global N8N_STATUS_URL
    N8N_STATUS_URL = url
    print(f"✅ N8N Status URL updated: {url[:50]}...")

print(f"\nConfig loaded:")
print(f"  NODE_NAME            : {NODE_NAME}")
print(f"  MODEL_NAME           : {MODEL_NAME}")
print(f"  BATCH_SIZE           : {BATCH_SIZE}")
print(f"  MAX_HOURS            : {MAX_HOURS}h")
print(f"  STATUS_UPDATE_INTERVAL: {STATUS_UPDATE_INTERVAL} articles")
print(f"  N8N_GET              : {N8N_GET_URL[:40]}...")
print(f"  N8N_STATUS           : {N8N_STATUS_URL[:40]}...")
print(f"\nTo change URL at runtime:")
print(f"  set_n8n_status_url('YOUR_NEW_URL')")

---
## 🤖 Cell 3 — Load Model
Takes 2–3 minutes. Loads the model in GPU float16 (half precision) for best performance on 16 GB GPU memory.

In [ ]:
# ============================================================
# CELL 3 — Load Qwen3-Embedding-8B model on GPU using float16
# ============================================================
from sentence_transformers import SentenceTransformer
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for best performance. Enable GPU accelerator and rerun this notebook.")

device = "cuda"
print(f"Device: {device}")
print(f"GPU memory before load: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

model = SentenceTransformer(
    MODEL_PATH,
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)
model = model.to(device)

torch.cuda.empty_cache()
used = torch.cuda.memory_allocated(0) / 1e9
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
print(f"GPU used: {used:.1f} GB  |  free: {free:.1f} GB")

# Verify model output dimensions
test_vector = model.encode(["stock market earnings report"], normalize_embeddings=True)
print(f"✅ Model ready — embedding shape: {test_vector.shape}")

if test_vector.shape[1] != 4096:
    print(f"⚠️  WARNING: Expected 4096 dimensions, got {test_vector.shape[1]}")
else:
    print("✅ Dimensions correct: 4096")

---
## 🚀 Cell 4 — Start Embedding
Runs until all articles are embedded or `MAX_HOURS` is reached.

Progress is saved automatically — if the session stops, just run again and it continues from where it left off.

In [ ]:
# ============================================================
# CELL 4 — Main embedding loop
# Fetches articles from n8n → embeds → saves vectors back.
# Safe to stop and resume at any time.
# Progress updates sent to n8n every STATUS_UPDATE_INTERVAL articles.
# ============================================================
import requests
import time
import torch
from datetime import datetime, timedelta

headers  = {"X-API-Key": N8N_API_KEY, "Content-Type": "application/json"}
total    = 0
errors   = 0
start    = time.time()
session_start_total = 0  # Track how many embedded in this session

print(f"Worker  : {NODE_NAME}")
print(f"Batch   : {BATCH_SIZE}")
print(f"Max time: {MAX_HOURS}h")
print(f"Status updates every: {STATUS_UPDATE_INTERVAL} articles")
print("-" * 80)
print(f"⏱️  Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 80)

def send_status_to_n8n(total_embedded, session_count, speed_per_min, speed_per_hour, 
                       estimated_stop_time, elapsed_minutes, errors_count, gpu_mem_gb, status="running"):
    """
    Send progress status to n8n webhook for database storage.
    """
    try:
        payload = {
            "node_name": NODE_NAME,
            "timestamp": datetime.now().isoformat(),
            "total_embedded": total_embedded,
            "embedded_in_session": session_count,
            "embedded_per_minute": round(speed_per_min, 2),
            "embedded_per_hour": round(speed_per_hour, 2),
            "estimated_stop_time": estimated_stop_time.isoformat(),
            "elapsed_time_minutes": round(elapsed_minutes, 1),
            "errors": errors_count,
            "status": status,
            "batch_size": BATCH_SIZE,
            "gpu_memory_used_gb": round(gpu_mem_gb, 2),
            "max_hours_limit": MAX_HOURS
        }

        resp = requests.post(
            N8N_STATUS_URL,
            json=payload,
            headers=headers,
            timeout=15
        )

        if resp.status_code in [200, 201]:
            print(f"  ✓ Status sent to n8n")
        else:
            print(f"  ⚠️  Status webhook returned {resp.status_code}")

    except Exception as e:
        print(f"  ⚠️  Failed to send status to n8n: {str(e)[:60]}")

while True:

    # --- Time limit ---
    elapsed_h = (time.time() - start) / 3600
    if elapsed_h >= MAX_HOURS:
        print(f"\n{'='*80}")
        print(f"⏱️  Time limit reached ({MAX_HOURS}h). Stopping cleanly.")
        print(f"Total embedded: {total:,}")
        print(f"Session count: {session_start_total:,}")
        print(f"Errors: {errors}")
        print(f"Stopped at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        send_status_to_n8n(
            total_embedded=total,
            session_count=session_start_total,
            speed_per_min=total / max((time.time() - start) / 60, 1),
            speed_per_hour=total / max((time.time() - start) / 3600, 1),
            estimated_stop_time=datetime.now(),
            elapsed_minutes=(time.time() - start) / 60,
            errors_count=errors,
            gpu_mem_gb=torch.cuda.memory_allocated(0) / 1e9,
            status="stopped"
        )
        break

    # --- Get batch from n8n ---
    try:
        resp = requests.post(
            N8N_GET_URL,
            json={"batch_size": BATCH_SIZE, "node_name": NODE_NAME},
            headers=headers,
            timeout=30
        )
        records = resp.json().get("records", [])
    except Exception as e:
        print(f"⚠️  GET error: {e}")
        errors += 1
        time.sleep(10)
        continue

    if not records:
        print(f"\n{'='*80}")
        print(f"✅ No more pending articles. All done!")
        print(f"Total embedded: {total:,}")
        print(f"Session count: {session_start_total:,}")
        print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}\n")
        send_status_to_n8n(
            total_embedded=total,
            session_count=session_start_total,
            speed_per_min=total / max((time.time() - start) / 60, 1),
            speed_per_hour=total / max((time.time() - start) / 3600, 1),
            estimated_stop_time=datetime.now(),
            elapsed_minutes=(time.time() - start) / 60,
            errors_count=errors,
            gpu_mem_gb=torch.cuda.memory_allocated(0) / 1e9,
            status="done"
        )
        break

    ids   = [r["id"] for r in records]
    texts = [r.get("description") or r.get("title") or "" for r in records]

    # --- Embed ---
    try:
        vectors = model.encode(
            texts,
            batch_size=1,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        torch.cuda.empty_cache()

    except RuntimeError as e:
        torch.cuda.empty_cache()
        print(f"⚠️  Embed error (id={ids}): {str(e)[:80]}")
        errors += 1
        time.sleep(5)
        continue

    # --- Save vectors to n8n ---
    try:
        payload = {"vectors": [
            {"id": ids[i], "vector": vectors[i].tolist(), "node_name": NODE_NAME}
            for i in range(len(ids))
        ]}
        requests.post(N8N_SAVE_URL, json=payload, headers=headers, timeout=60)
    except Exception as e:
        print(f"⚠️  SAVE error: {e}")
        errors += 1
        time.sleep(5)
        continue

    # --- Update counters and stats ---
    total     += len(records)
    session_start_total += len(records)
    elapsed_s  = time.time() - start
    elapsed_m  = elapsed_s / 60
    elapsed_h  = elapsed_s / 3600
    
    # Calculate speeds
    speed_per_min = total / elapsed_m if elapsed_m > 0 else 0
    speed_per_hour = total / elapsed_h if elapsed_h > 0 else 0
    
    # Calculate estimated stop time
    remaining_h = max(MAX_HOURS - elapsed_h, 0)
    estimated_stop_time = datetime.now() + timedelta(hours=remaining_h)
    
    # Get GPU memory usage
    gpu_mem_gb = torch.cuda.memory_allocated(0) / 1e9
    
    # --- Send status update every N articles ---
    if session_start_total % STATUS_UPDATE_INTERVAL == 0:
        print(f"\n📊 STATUS UPDATE (articles embedded: {session_start_total:,})")
        print(f"  ├─ Total all-time      : {total:,}")
        print(f"  ├─ Speed               : {speed_per_min:.1f} art/min | {speed_per_hour:.1f} art/hour")
        print(f"  ├─ Elapsed time        : {elapsed_m:.1f} min ({elapsed_h:.2f}h)")
        print(f"  ├─ Estimated stop time : {estimated_stop_time.strftime('%H:%M')}")
        print(f"  ├─ GPU memory used     : {gpu_mem_gb:.2f} GB")
        print(f"  └─ Errors              : {errors}")
        
        send_status_to_n8n(
            total_embedded=total,
            session_count=session_start_total,
            speed_per_min=speed_per_min,
            speed_per_hour=speed_per_hour,
            estimated_stop_time=estimated_stop_time,
            elapsed_minutes=elapsed_m,
            errors_count=errors,
            gpu_mem_gb=gpu_mem_gb,
            status="running"
        )
        print()
